# 🚀 Hybrid Retrieval - The Production Standard

## What is Hybrid Retrieval?

**Hybrid retrieval combines sparse (keyword) and dense (semantic) methods to get the best of both worlds.**

### The Power Combo:

```
Query: "machine learning algorithms"
    ↓
┌─────────────────┐         ┌─────────────────┐
│  BM25 (Sparse)  │         │ Dense (Semantic)│
│  Exact matches  │         │ Similar meaning │
└─────────────────┘         └─────────────────┘
         ↓                           ↓
    Scores: [0.8, 0.3, 0.6]    Scores: [0.5, 0.9, 0.4]
         ↓                           ↓
         └───────────┬───────────────┘
                     ↓
              FUSION (Combine)
                     ↓
         Final: [0.7, 0.6, 0.5]
```

![Hybrid Retrieval Architecture](./images/hybrid_retrieval_arch.png)
*Figure 1: Hybrid retrieval pipeline combining BM25 and dense retrieval*

## Why Hybrid Wins 🏆

### The Problem:
- **BM25 alone**: Misses semantic matches ("car" ≠ "automobile")
- **Dense alone**: Misses exact matches ("COVID-19" ≠ "coronavirus")

### The Solution:
- **Hybrid**: Gets BOTH exact AND semantic matches! ✨

### Real-World Evidence:
```
Google, Bing, Elasticsearch all use hybrid retrieval!
```

## Fusion Methods

### 1. **Reciprocal Rank Fusion (RRF)** ⭐ Most Popular
```python
RRF_score = Σ 1/(k + rank_i)
where k = 60 (constant)
```

### 2. **Linear Combination**
```python
score = α × sparse_score + (1-α) × dense_score
where α ∈ [0, 1]
```

### 3. **Convex Combination** (Normalized)
```python
score = α × norm(sparse) + (1-α) × norm(dense)
```

## 1. Simple Hybrid Retrieval

In [1]:
# Install dependencies
# !pip install sentence-transformers rank-bm25 scikit-learn

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
import numpy as np
import nltk

nltk.download('punkt', quiet=True)

# Sample documents
documents = [
    "Python is a programming language used for data science and machine learning.",
    "Machine learning algorithms can predict outcomes from historical data.",
    "Natural language processing helps computers understand human language.",
    "Deep neural networks are the foundation of modern AI systems.",
    "RAG combines retrieval and generation for better language model responses.",
    "Vector databases efficiently store and query high-dimensional embeddings.",
]

print(f"Documents: {len(documents)}")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc}")

Documents: 6
1. Python is a programming language used for data science and machine learning.
2. Machine learning algorithms can predict outcomes from historical data.
3. Natural language processing helps computers understand human language.
4. Deep neural networks are the foundation of modern AI systems.
5. RAG combines retrieval and generation for better language model responses.
6. Vector databases efficiently store and query high-dimensional embeddings.


In [2]:
# Setup retrievers

# 1. BM25 (Sparse)
tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# 2. Dense (Semantic)
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)

print("✅ Both retrievers initialized!")
print(f"   BM25: {len(tokenized_docs)} docs tokenized")
print(f"   Dense: {doc_embeddings.shape} embeddings")

✅ Both retrievers initialized!
   BM25: 6 docs tokenized
   Dense: (6, 384) embeddings


In [3]:
# Compare individual methods
query = "Python machine learning"

# BM25 scores
bm25_scores = bm25.get_scores(word_tokenize(query.lower()))

# Dense scores
query_embedding = dense_model.encode(query)
dense_scores = cosine_similarity([query_embedding], doc_embeddings)[0]

print(f"Query: '{query}'\n")
print(f"{'Doc':<5} {'BM25':<12} {'Dense':<12} {'Document'}")
print("="*80)

for i, doc in enumerate(documents):
    print(f"{i+1:<5} {bm25_scores[i]:<12.4f} {dense_scores[i]:<12.4f} {doc[:45]}...")

print("\n💡 Different scores! Let's combine them...")

Query: 'Python machine learning'

Doc   BM25         Dense        Document
1     2.2354       0.5785       Python is a programming language used for dat...
2     1.2013       0.5048       Machine learning algorithms can predict outco...
3     0.0000       0.2773       Natural language processing helps computers u...
4     0.0000       0.2957       Deep neural networks are the foundation of mo...
5     0.0000       0.2041       RAG combines retrieval and generation for bet...
6     0.0000       0.1456       Vector databases efficiently store and query ...

💡 Different scores! Let's combine them...


## 2. Reciprocal Rank Fusion (RRF)

In [4]:
def reciprocal_rank_fusion(sparse_scores, dense_scores, k=60):
    """
    Reciprocal Rank Fusion (RRF)
    
    Formula: RRF_score = Σ 1/(k + rank_i)
    
    Parameters:
    k: Constant to prevent division by zero and control fusion (default 60)
    """
    # Get rankings (0 = best)
    sparse_ranks = np.argsort(sparse_scores)[::-1]
    dense_ranks = np.argsort(dense_scores)[::-1]
    
    # Calculate RRF scores
    rrf_scores = np.zeros(len(sparse_scores))
    
    for rank, idx in enumerate(sparse_ranks):
        rrf_scores[idx] += 1 / (k + rank + 1)
    
    for rank, idx in enumerate(dense_ranks):
        rrf_scores[idx] += 1 / (k + rank + 1)
    
    return rrf_scores

# Apply RRF
rrf_scores = reciprocal_rank_fusion(bm25_scores, dense_scores)

# Get top results
top_indices = np.argsort(rrf_scores)[::-1][:3]

print("Reciprocal Rank Fusion Results:\n")
print(f"Query: '{query}'\n")

for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. [RRF: {rrf_scores[idx]:.4f}] {documents[idx]}")
    print(f"   BM25: {bm25_scores[idx]:.4f}, Dense: {dense_scores[idx]:.4f}\n")

Reciprocal Rank Fusion Results:

Query: 'Python machine learning'

1. [RRF: 0.0328] Python is a programming language used for data science and machine learning.
   BM25: 2.2354, Dense: 0.5785

2. [RRF: 0.0323] Machine learning algorithms can predict outcomes from historical data.
   BM25: 1.2013, Dense: 0.5048

3. [RRF: 0.0313] Deep neural networks are the foundation of modern AI systems.
   BM25: 0.0000, Dense: 0.2957



## 3. Linear Combination

In [5]:
def linear_combination(sparse_scores, dense_scores, alpha=0.5):
    """
    Linear combination with normalization
    
    score = α × norm(sparse) + (1-α) × norm(dense)
    
    Parameters:
    alpha: Weight for sparse scores (0-1)
    """
    # Normalize scores to [0, 1]
    sparse_norm = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-6)
    dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-6)
    
    # Combine
    combined = alpha * sparse_norm + (1 - alpha) * dense_norm
    
    return combined

# Test different alpha values
alpha_values = [0.0, 0.3, 0.5, 0.7, 1.0]

print(f"Linear Combination with Different α:\n")
print(f"Query: '{query}'\n")
print(f"{'α':<8} {'Top Document'}")
print("="*80)

for alpha in alpha_values:
    combined_scores = linear_combination(bm25_scores, dense_scores, alpha=alpha)
    top_idx = np.argmax(combined_scores)
    
    weight_type = "Dense only" if alpha == 0 else "BM25 only" if alpha == 1 else f"Hybrid ({int(alpha*100)}% sparse)"
    print(f"{alpha:<8} [{weight_type}] {documents[top_idx][:60]}...")

print("\n💡 α=0.5 gives equal weight to both methods")

Linear Combination with Different α:

Query: 'Python machine learning'

α        Top Document
0.0      [Dense only] Python is a programming language used for data science and m...
0.3      [Hybrid (30% sparse)] Python is a programming language used for data science and m...
0.5      [Hybrid (50% sparse)] Python is a programming language used for data science and m...
0.7      [Hybrid (70% sparse)] Python is a programming language used for data science and m...
1.0      [BM25 only] Python is a programming language used for data science and m...

💡 α=0.5 gives equal weight to both methods


## 4. Complete Hybrid Retriever Class

In [6]:
from typing import List, Dict, Literal

class HybridRetriever:
    def __init__(self, 
                 dense_model_name='all-MiniLM-L6-v2',
                 fusion_method: Literal['rrf', 'linear'] = 'rrf',
                 alpha=0.5,
                 k=60):
        """
        Hybrid Retriever combining BM25 and dense retrieval
        
        Parameters:
        - fusion_method: 'rrf' or 'linear'
        - alpha: Weight for sparse (used in linear fusion)
        - k: RRF constant (used in RRF fusion)
        """
        self.dense_model = SentenceTransformer(dense_model_name)
        self.fusion_method = fusion_method
        self.alpha = alpha
        self.k = k
        
        self.documents = []
        self.bm25 = None
        self.doc_embeddings = None
        
    def index(self, documents: List[str]):
        """Index documents for both sparse and dense retrieval"""
        self.documents = documents
        
        # BM25 index
        tokenized = [word_tokenize(doc.lower()) for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
        
        # Dense index
        self.doc_embeddings = self.dense_model.encode(
            documents, 
            show_progress_bar=True
        )
        
        print(f"✅ Indexed {len(documents)} documents (hybrid)")
    
    def _reciprocal_rank_fusion(self, sparse_scores, dense_scores):
        """RRF fusion"""
        sparse_ranks = np.argsort(sparse_scores)[::-1]
        dense_ranks = np.argsort(dense_scores)[::-1]
        
        rrf_scores = np.zeros(len(sparse_scores))
        
        for rank, idx in enumerate(sparse_ranks):
            rrf_scores[idx] += 1 / (self.k + rank + 1)
        
        for rank, idx in enumerate(dense_ranks):
            rrf_scores[idx] += 1 / (self.k + rank + 1)
        
        return rrf_scores
    
    def _linear_combination(self, sparse_scores, dense_scores):
        """Linear combination fusion"""
        sparse_norm = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-6)
        dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-6)
        
        return self.alpha * sparse_norm + (1 - self.alpha) * dense_norm
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Hybrid search"""
        # Sparse scores
        tokenized_query = word_tokenize(query.lower())
        sparse_scores = self.bm25.get_scores(tokenized_query)
        
        # Dense scores
        query_embedding = self.dense_model.encode(query)
        dense_scores = cosine_similarity([query_embedding], self.doc_embeddings)[0]
        
        # Fusion
        if self.fusion_method == 'rrf':
            final_scores = self._reciprocal_rank_fusion(sparse_scores, dense_scores)
        else:
            final_scores = self._linear_combination(sparse_scores, dense_scores)
        
        # Get top-k
        top_indices = np.argsort(final_scores)[::-1][:top_k]
        
        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                'rank': rank + 1,
                'score': float(final_scores[idx]),
                'sparse_score': float(sparse_scores[idx]),
                'dense_score': float(dense_scores[idx]),
                'document': self.documents[idx],
                'index': int(idx)
            })
        
        return results

# Test hybrid retriever
hybrid = HybridRetriever(fusion_method='rrf')
hybrid.index(documents)

query = "Python machine learning"
results = hybrid.search(query, top_k=3)

print(f"\nHybrid Search Results:\n")
print(f"Query: '{query}'\n")

for r in results:
    print(f"{r['rank']}. [Hybrid: {r['score']:.4f}]")
    print(f"   BM25: {r['sparse_score']:.4f}, Dense: {r['dense_score']:.4f}")
    print(f"   {r['document']}\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 6 documents (hybrid)

Hybrid Search Results:

Query: 'Python machine learning'

1. [Hybrid: 0.0328]
   BM25: 2.2354, Dense: 0.5785
   Python is a programming language used for data science and machine learning.

2. [Hybrid: 0.0323]
   BM25: 1.2013, Dense: 0.5048
   Machine learning algorithms can predict outcomes from historical data.

3. [Hybrid: 0.0313]
   BM25: 0.0000, Dense: 0.2957
   Deep neural networks are the foundation of modern AI systems.



## 5. Comparing All Three Methods

In [7]:
# Test queries showcasing different retrieval strengths
test_queries = [
    ("Python programming", "Exact keyword match"),
    ("neural network architectures", "Semantic understanding"),
    ("NLP and text understanding", "Abbreviation + semantic"),
]

print("Method Comparison on Different Query Types:\n")
print("="*90)

for query, desc in test_queries:
    print(f"\nQuery: '{query}' ({desc})\n")
    
    # BM25 only
    bm25_scores = bm25.get_scores(word_tokenize(query.lower()))
    bm25_top = documents[np.argmax(bm25_scores)]
    
    # Dense only
    query_emb = dense_model.encode(query)
    dense_scores = cosine_similarity([query_emb], doc_embeddings)[0]
    dense_top = documents[np.argmax(dense_scores)]
    
    # Hybrid
    hybrid_results = hybrid.search(query, top_k=1)
    hybrid_top = hybrid_results[0]['document']
    
    print(f"BM25:   {bm25_top}")
    print(f"Dense:  {dense_top}")
    print(f"Hybrid: {hybrid_top}")
    
    # Check agreement
    if bm25_top == dense_top == hybrid_top:
        print("✅ All methods agree!")
    elif hybrid_top in [bm25_top, dense_top]:
        print("⚡ Hybrid picks best individual result")
    else:
        print("🔀 Hybrid finds different result")

Method Comparison on Different Query Types:


Query: 'Python programming' (Exact keyword match)

BM25:   Python is a programming language used for data science and machine learning.
Dense:  Python is a programming language used for data science and machine learning.
Hybrid: Python is a programming language used for data science and machine learning.
✅ All methods agree!

Query: 'neural network architectures' (Semantic understanding)

BM25:   Deep neural networks are the foundation of modern AI systems.
Dense:  Deep neural networks are the foundation of modern AI systems.
Hybrid: Deep neural networks are the foundation of modern AI systems.
✅ All methods agree!

Query: 'NLP and text understanding' (Abbreviation + semantic)

BM25:   Python is a programming language used for data science and machine learning.
Dense:  Natural language processing helps computers understand human language.
Hybrid: RAG combines retrieval and generation for better language model responses.
🔀 Hybrid finds diffe

## 6. Tuning Fusion Parameters

In [8]:
# Compare RRF k values
query = "machine learning algorithms"
k_values = [10, 30, 60, 100, 200]

print("Effect of RRF k Parameter:\n")
print(f"Query: '{query}'\n")
print(f"{'k':<8} {'Top Document'}")
print("="*80)

for k in k_values:
    retriever = HybridRetriever(fusion_method='rrf', k=k)
    retriever.index(documents)
    results = retriever.search(query, top_k=1)
    
    print(f"{k:<8} {results[0]['document'][:65]}...")

print("\n💡 k controls fusion strength:")
print("   Lower k → Ranks matter more")
print("   Higher k → Scores matter more")
print("   Default k=60 works well for most cases")

Effect of RRF k Parameter:

Query: 'machine learning algorithms'

k        Top Document


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 6 documents (hybrid)
10       Machine learning algorithms can predict outcomes from historical ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 6 documents (hybrid)
30       Machine learning algorithms can predict outcomes from historical ...


...

✅ Indexed 6 documents (hybrid)
60       Machine learning algorithms can predict outcomes from historical ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 6 documents (hybrid)
100      Machine learning algorithms can predict outcomes from historical ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 6 documents (hybrid)
200      Machine learning algorithms can predict outcomes from historical ...

💡 k controls fusion strength:
   Lower k → Ranks matter more
   Higher k → Scores matter more
   Default k=60 works well for most cases


## 7. Production Hybrid System

In [9]:
import pickle
from pathlib import Path

class ProductionHybridRetriever:
    def __init__(self,
                 dense_model_name='all-MiniLM-L6-v2',
                 fusion_method='rrf',
                 alpha=0.5,
                 k=60,
                 bm25_k1=1.5,
                 bm25_b=0.75):
        
        self.dense_model = SentenceTransformer(dense_model_name)
        self.fusion_method = fusion_method
        self.alpha = alpha
        self.k = k
        self.bm25_k1 = bm25_k1
        self.bm25_b = bm25_b
        
        self.documents = []
        self.metadata = []
        self.bm25 = None
        self.doc_embeddings = None
    
    def index(self, documents: List[str], metadata: List[Dict] = None):
        """Index with optional metadata"""
        self.documents = documents
        self.metadata = metadata if metadata else [{} for _ in documents]
        
        # BM25
        tokenized = [word_tokenize(doc.lower()) for doc in documents]
        self.bm25 = BM25Okapi(tokenized, k1=self.bm25_k1, b=self.bm25_b)
        
        # Dense
        self.doc_embeddings = self.dense_model.encode(
            documents,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        
        print(f"✅ Indexed {len(documents)} documents (production hybrid)")
    
    def _fuse_scores(self, sparse_scores, dense_scores):
        if self.fusion_method == 'rrf':
            sparse_ranks = np.argsort(sparse_scores)[::-1]
            dense_ranks = np.argsort(dense_scores)[::-1]
            
            scores = np.zeros(len(sparse_scores))
            for rank, idx in enumerate(sparse_ranks):
                scores[idx] += 1 / (self.k + rank + 1)
            for rank, idx in enumerate(dense_ranks):
                scores[idx] += 1 / (self.k + rank + 1)
            return scores
        else:
            sparse_norm = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-6)
            dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-6)
            return self.alpha * sparse_norm + (1 - self.alpha) * dense_norm
    
    def search(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
        # Get scores
        sparse_scores = self.bm25.get_scores(word_tokenize(query.lower()))
        query_emb = self.dense_model.encode(query, normalize_embeddings=True)
        dense_scores = np.dot(self.doc_embeddings, query_emb)
        
        # Fuse
        final_scores = self._fuse_scores(sparse_scores, dense_scores)
        
        # Filter and rank
        valid_indices = np.where(final_scores >= score_threshold)[0]
        valid_scores = final_scores[valid_indices]
        
        top_k = min(top_k, len(valid_scores))
        top_positions = np.argsort(valid_scores)[::-1][:top_k]
        top_indices = valid_indices[top_positions]
        
        return [{
            'rank': rank + 1,
            'score': float(final_scores[idx]),
            'sparse_score': float(sparse_scores[idx]),
            'dense_score': float(dense_scores[idx]),
            'document': self.documents[idx],
            'metadata': self.metadata[idx],
            'index': int(idx)
        } for rank, idx in enumerate(top_indices)]
    
    def save(self, directory: str):
        """Save index to directory"""
        path = Path(directory)
        path.mkdir(exist_ok=True)
        
        # Save config
        config = {
            'fusion_method': self.fusion_method,
            'alpha': self.alpha,
            'k': self.k,
            'bm25_k1': self.bm25_k1,
            'bm25_b': self.bm25_b
        }
        
        with open(path / 'config.pkl', 'wb') as f:
            pickle.dump(config, f)
        
        # Save data
        np.savez(
            path / 'data.npz',
            embeddings=self.doc_embeddings,
            documents=self.documents,
            metadata=self.metadata
        )
        
        # Save BM25
        with open(path / 'bm25.pkl', 'wb') as f:
            pickle.dump(self.bm25, f)
        
        print(f"✅ Saved to {directory}/")
    
    def load(self, directory: str):
        """Load index from directory"""
        path = Path(directory)
        
        # Load config
        with open(path / 'config.pkl', 'rb') as f:
            config = pickle.load(f)
        
        self.fusion_method = config['fusion_method']
        self.alpha = config['alpha']
        self.k = config['k']
        self.bm25_k1 = config['bm25_k1']
        self.bm25_b = config['bm25_b']
        
        # Load data
        data = np.load(path / 'data.npz', allow_pickle=True)
        self.doc_embeddings = data['embeddings']
        self.documents = data['documents'].tolist()
        self.metadata = data['metadata'].tolist()
        
        # Load BM25
        with open(path / 'bm25.pkl', 'rb') as f:
            self.bm25 = pickle.load(f)
        
        print(f"✅ Loaded from {directory}/")

# Test production system
prod_hybrid = ProductionHybridRetriever(fusion_method='rrf', k=60)

docs_with_meta = [
    ("RAG combines retrieval with generation", {"type": "definition", "topic": "RAG"}),
    ("BM25 is a ranking function for sparse retrieval", {"type": "definition", "topic": "BM25"}),
    ("Dense embeddings capture semantic meaning", {"type": "concept", "topic": "embeddings"}),
    ("Hybrid retrieval gets best of both worlds", {"type": "concept", "topic": "hybrid"}),
]

docs = [d[0] for d in docs_with_meta]
meta = [d[1] for d in docs_with_meta]

prod_hybrid.index(docs, metadata=meta)

results = prod_hybrid.search("What combines keyword and semantic search?", top_k=2)

print("\nProduction Search:")
for r in results:
    print(f"\n{r['rank']}. {r['document']}")
    print(f"   Score: {r['score']:.4f} (BM25: {r['sparse_score']:.4f}, Dense: {r['dense_score']:.4f})")
    print(f"   {r['metadata']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 4 documents (production hybrid)

Production Search:

1. Dense embeddings capture semantic meaning
   Score: 0.0325 (BM25: 0.9311, Dense: 0.3561)
   {'type': 'concept', 'topic': 'embeddings'}

2. Hybrid retrieval gets best of both worlds
   Score: 0.0323 (BM25: 0.0000, Dense: 0.4150)
   {'type': 'concept', 'topic': 'hybrid'}


## 8. Real-World RAG Example

![Hybrid RAG Pipeline](./images/hybrid_rag_pipeline.png)
*Figure 2: Complete hybrid RAG pipeline from query to answer generation*

In [10]:
# Complete RAG knowledge base
rag_kb = [
    "RAG (Retrieval-Augmented Generation) enhances LLMs by retrieving relevant context before generation.",
    "BM25 algorithm ranks documents using term frequency and inverse document frequency statistics.",
    "Dense retrieval converts text to embeddings and uses cosine similarity for semantic matching.",
    "Hybrid retrieval combines BM25 (exact matching) with dense retrieval (semantic understanding).",
    "Reciprocal Rank Fusion (RRF) is the most popular method to combine sparse and dense scores.",
    "Vector databases like Pinecone, Weaviate, and Chroma store embeddings for fast similarity search.",
    "The k1 parameter in BM25 controls term frequency saturation, typically between 1.2 and 2.0.",
    "Embedding models like BERT, E5, and BGE convert text into dense vector representations.",
    "Production RAG systems often use hybrid retrieval for better recall and precision.",
    "Re-ranking can further improve hybrid retrieval by reordering top-k results with a cross-encoder.",
]

# Build hybrid RAG retriever
rag_hybrid = ProductionHybridRetriever(fusion_method='rrf', k=60)
rag_hybrid.index(rag_kb)

# Test questions
questions = [
    "How does hybrid retrieval work?",
    "What is RRF in RAG systems?",
    "Best parameters for BM25?",
    "Which databases for RAG?",
]

print("Hybrid RAG Knowledge Base Q&A:\n")
print("="*90)

for question in questions:
    results = rag_hybrid.search(question, top_k=2)
    
    print(f"\n❓ {question}")
    print(f"\n📄 Retrieved Context (Hybrid):")
    for r in results:
        method = "BM25" if r['sparse_score'] > r['dense_score'] else "Dense"
        print(f"   [{r['score']:.3f}] ({method} dominant) {r['document']}")

print("\n💡 Hybrid retrieval provides robust results across different query types!")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 10 documents (production hybrid)
Hybrid RAG Knowledge Base Q&A:


❓ How does hybrid retrieval work?

📄 Retrieved Context (Hybrid):
   [0.033] (BM25 dominant) Hybrid retrieval combines BM25 (exact matching) with dense retrieval (semantic understanding).
   [0.032] (BM25 dominant) Production RAG systems often use hybrid retrieval for better recall and precision.

❓ What is RRF in RAG systems?

📄 Retrieved Context (Hybrid):
   [0.033] (BM25 dominant) Production RAG systems often use hybrid retrieval for better recall and precision.
   [0.033] (BM25 dominant) Reciprocal Rank Fusion (RRF) is the most popular method to combine sparse and dense scores.

❓ Best parameters for BM25?

📄 Retrieved Context (Hybrid):
   [0.032] (BM25 dominant) BM25 algorithm ranks documents using term frequency and inverse document frequency statistics.
   [0.032] (BM25 dominant) The k1 parameter in BM25 controls term frequency saturation, typically between 1.2 and 2.0.

❓ Which databases for RAG?

📄 Retr

## Key Takeaways 🎯

### ✅ Why Hybrid is Production Standard:

1. **Best Recall** - Catches both exact and semantic matches
2. **Robust** - Works well across diverse query types  
3. **Industry Proven** - Used by Google, Bing, Elasticsearch
4. **Balanced** - Combines speed of BM25 with power of dense
5. **Flexible** - Tunable parameters for your domain

### 🔑 Fusion Methods:

| Method | When to Use | Parameters | Pros | Cons |
|--------|-------------|------------|------|------|
| **RRF** | Default choice | k=60 | Simple, effective | Less control |
| **Linear** | Need control | α=0.5 | Tunable weights | Requires tuning |
| **Convex** | Normalized scores | α=0.5 | Stable | Similar to linear |

**Recommendation: Start with RRF (k=60), it works great!**

### 📊 Parameter Guide:

```python
# RRF Parameters
k = 60  # Standard (10-100 range)
- Lower k (10-30): Ranks matter more
- Higher k (100+): Scores matter more

# Linear Combination
alpha = 0.5  # Equal weight (0-1 range)
- α = 0.0: Dense only
- α = 0.3: Mostly dense
- α = 0.5: Balanced
- α = 0.7: Mostly sparse
- α = 1.0: Sparse only

# BM25 (within hybrid)
k1 = 1.5  # Term saturation
b = 0.75  # Length normalization
```

### 🚀 Production Setup:

```python
# Recommended configuration
retriever = ProductionHybridRetriever(
    dense_model_name='all-MiniLM-L6-v2',  # Fast & good
    fusion_method='rrf',  # Proven method
    k=60,  # Standard RRF constant
    bm25_k1=1.5,  # Standard BM25
    bm25_b=0.75   # Standard normalization
)

# Index your documents
retriever.index(documents, metadata)

# Search
results = retriever.search(query, top_k=5)

# Save for later
retriever.save('hybrid_index/')
```

### 💡 Best Practices:

1. **Start with RRF**: Works well out of the box
2. **Monitor both scores**: Track sparse vs dense contribution
3. **A/B test fusion**: Compare RRF vs linear on your data
4. **Save indexes**: Avoid re-indexing in production
5. **Add metadata**: Track source, date, category, etc.

### 🎯 When Each Method Shines:

**Hybrid excels at:**
- ✅ Mixed query types (exact + semantic)
- ✅ Robustness across domains
- ✅ Production RAG systems
- ✅ E-commerce search
- ✅ Enterprise search

**BM25 alone for:**
- Exact product codes/IDs
- Legal/regulatory text
- Code search

**Dense alone for:**
- Semantic Q&A
- Conceptual search
- Multilingual search

### 📈 Performance Impact:

```
Sparse (BM25):     10-50ms per query
Dense:             100-500ms per query
Hybrid (RRF):      110-550ms per query

Overhead: ~10% for fusion
Benefit: 10-30% better relevance

Worth it! ✅
```

### 🔬 Advanced: Multi-Stage Retrieval

```python
# Stage 1: Hybrid retrieval (cast wide net)
candidates = hybrid.search(query, top_k=100)

# Stage 2: Re-rank top candidates
reranked = reranker.rerank(query, candidates, top_k=10)

# Stage 3: Final selection
final = select_diverse(reranked, top_k=5)
```

## Next Steps 🔜

Move to `04_Vector_Databases.ipynb` to learn production storage!

We'll cover Pinecone, Weaviate, Chroma, and more! 🗄️